# 01 · Data preprocessing

Load FD001 to FD004, drop six constant sensors, build the RUL and the failure label (RUL <= 20 cycles), and split by engine into train, validation A, validation B and test.

- **Reads:** NASA C-MAPSS `train_FD00x.txt`, `test_FD00x.txt`, `RUL_FD00x.txt`
- **Writes:** per-split parquet files, `sensor_metadata.json`, `split_metadata.json`, `dataset_manifest.*`

Written for Kaggle (`/kaggle/input/...`, `/kaggle/working/...`; the Kaggle account name is replaced by `<kaggle-user>`). Outputs are kept as saved. Read `docs/PIPELINE.md` for the order and for what is missing.

In [1]:
# ==========================================================
# NB01 : DATA PREPROCESSING + ENGINE SPLITS
# ==========================================================

import json
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

SEED = 42

np.random.seed(SEED)
random.seed(SEED)

WORKING_DIR = Path("/kaggle/working")

DATA_DIR = Path(
    "/kaggle/input/datasets/<kaggle-user>/nasa-cmaps"
)

OUTPUT_ROOT = WORKING_DIR / "NB-1-AGAIN-outputs"

PROCESSED_DIR = OUTPUT_ROOT / "processed"
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs"

for d in [PROCESSED_DIR, METADATA_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASETS = ["FD001", "FD002", "FD003", "FD004"]

RUL_HORIZON = 20

TRAIN_RATIO = 0.70
VAL_A_RATIO = 0.10
VAL_B_RATIO = 0.05
TEST_RATIO = 0.15

assert TRAIN_RATIO + VAL_A_RATIO + VAL_B_RATIO + TEST_RATIO == 1.0

print("✅ CELL 1 DONE")

✅ CELL 1 DONE


In [2]:
ALL_SENSORS = [f"sensor_{i}" for i in range(1, 22)]

SENSORS_REMOVED = [
    "sensor_1",
    "sensor_5",
    "sensor_10",
    "sensor_16",
    "sensor_18",
    "sensor_19"
]

ACTIVE_SENSORS = [
    s for s in ALL_SENSORS
    if s not in SENSORS_REMOVED
]

assert len(ACTIVE_SENSORS) == 15

OPERATING_SETTINGS = [
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
]

BASE_FEATURE_COLS = OPERATING_SETTINGS + ACTIVE_SENSORS

assert len(BASE_FEATURE_COLS) == 18

sensor_metadata = {
    "all_sensors": ALL_SENSORS,
    "sensors_removed": SENSORS_REMOVED,
    "active_sensors": ACTIVE_SENSORS,
    "operating_settings": OPERATING_SETTINGS,
    "num_active_sensors": len(ACTIVE_SENSORS),
    "num_base_features": len(BASE_FEATURE_COLS),
    "rul_horizon": RUL_HORIZON
}

with open(METADATA_DIR / "sensor_metadata.json", "w") as f:
    json.dump(sensor_metadata, f, indent=4)

print("Active sensors:", len(ACTIVE_SENSORS))
print("Base features:", len(BASE_FEATURE_COLS))

print("✅ CELL 2 DONE")

Active sensors: 15
Base features: 18
✅ CELL 2 DONE


In [3]:
COLUMNS = [
    "engine_id",
    "cycle",
    "op_setting_1",
    "op_setting_2",
    "op_setting_3"
]

COLUMNS += ALL_SENSORS

assert len(COLUMNS) == 26

print("✅ CELL 3 DONE")

✅ CELL 3 DONE


In [4]:
def load_dataset(dataset_id):

    train_path = DATA_DIR / f"train_{dataset_id}.txt"
    test_path = DATA_DIR / f"test_{dataset_id}.txt"
    rul_path = DATA_DIR / f"RUL_{dataset_id}.txt"

    train_df = pd.read_csv(
        train_path,
        sep=r"\s+",
        header=None
    ).dropna(axis=1)

    test_df = pd.read_csv(
        test_path,
        sep=r"\s+",
        header=None
    ).dropna(axis=1)

    rul_df = pd.read_csv(
        rul_path,
        sep=r"\s+",
        header=None
    )

    train_df.columns = COLUMNS
    test_df.columns = COLUMNS

    rul_df.columns = ["remaining_rul"]

    return train_df, test_df, rul_df


def add_rul(df):

    max_cycles = (
        df.groupby("engine_id")["cycle"]
        .max()
        .rename("max_cycle")
    )

    df = df.merge(max_cycles, on="engine_id")

    df["rul"] = df["max_cycle"] - df["cycle"]

    df.drop(columns=["max_cycle"], inplace=True)

    return df


def add_binary_label(df):

    df["failure_within_horizon"] = (
        df["rul"] <= RUL_HORIZON
    ).astype(int)

    return df


def keep_required_columns(df):

    required = (
        ["engine_id", "cycle"]
        + OPERATING_SETTINGS
        + ACTIVE_SENSORS
        + ["rul", "failure_within_horizon"]
    )

    return df[required].copy()

In [5]:
def create_splits(df):

    engines = sorted(df["engine_id"].unique())

    train_ids, temp_ids = train_test_split(
        engines,
        test_size=0.30,
        random_state=SEED,
        shuffle=True
    )

    val_a_ids, temp_ids = train_test_split(
        temp_ids,
        test_size=0.6667,
        random_state=SEED,
        shuffle=True
    )

    val_b_ids, test_ids = train_test_split(
        temp_ids,
        test_size=0.75,
        random_state=SEED,
        shuffle=True
    )

    return {
        "train": train_ids,
        "val_a": val_a_ids,
        "val_b": val_b_ids,
        "test": test_ids
    }

print("✅ CELL 5 DONE")

✅ CELL 5 DONE


In [6]:
manifest = []
split_metadata = {}

for dataset_id in DATASETS:

    print(f"\nProcessing {dataset_id}")

    train_df, test_df, rul_df = load_dataset(dataset_id)

    train_df = add_rul(train_df)

    train_df = add_binary_label(train_df)

    train_df = keep_required_columns(train_df)

    splits = create_splits(train_df)

    split_metadata[dataset_id] = {}

    for split_name, engine_ids in splits.items():

        split_df = train_df[
            train_df["engine_id"].isin(engine_ids)
        ].copy()

        output_path = (
            PROCESSED_DIR /
            f"{dataset_id}_{split_name}.parquet"
        )

        split_df.to_parquet(
            output_path,
            index=False
        )

        split_metadata[dataset_id][split_name] = {
            "engines": len(engine_ids),
            "rows": len(split_df)
        }

        print(
            f"{split_name:<6}"
            f" | engines={len(engine_ids):<4}"
            f" | rows={len(split_df):<7}"
        )

    manifest.append({
        "dataset": dataset_id,
        "active_sensors": len(ACTIVE_SENSORS),
        "base_features": len(BASE_FEATURE_COLS),
        "positive_rate": round(
            train_df["failure_within_horizon"].mean(),
            4
        )
    })

print("\n✅ CELL 6 DONE")


Processing FD001
train  | engines=70   | rows=14507  
val_a  | engines=9    | rows=1759   
val_b  | engines=5    | rows=1007   
test   | engines=16   | rows=3358   

Processing FD002
train  | engines=182  | rows=38255  
val_a  | engines=25   | rows=5002   
val_b  | engines=13   | rows=2510   
test   | engines=40   | rows=7992   

Processing FD003
train  | engines=70   | rows=17187  
val_a  | engines=9    | rows=2007   
val_b  | engines=5    | rows=1245   
test   | engines=16   | rows=4281   

Processing FD004
train  | engines=174  | rows=42834  
val_a  | engines=24   | rows=5974   
val_b  | engines=12   | rows=3066   
test   | engines=39   | rows=9375   

✅ CELL 6 DONE


In [7]:
manifest_df = pd.DataFrame(manifest)

manifest_df.to_csv(
    METADATA_DIR / "dataset_manifest.csv",
    index=False
)

with open(
    METADATA_DIR / "dataset_manifest.json",
    "w"
) as f:
    json.dump(manifest, f, indent=4)

with open(
    METADATA_DIR / "split_metadata.json",
    "w"
) as f:
    json.dump(split_metadata, f, indent=4)

print("✅ CELL 7 DONE")

✅ CELL 7 DONE


In [8]:
files = list(PROCESSED_DIR.glob("*.parquet"))

assert len(files) == 16

for file in files:

    df = pd.read_parquet(file)

    assert len(ACTIVE_SENSORS) == 15

    expected_cols = (
        ["engine_id", "cycle"]
        + OPERATING_SETTINGS
        + ACTIVE_SENSORS
        + ["rul", "failure_within_horizon"]
    )

    assert list(df.columns) == expected_cols

print("Processed files:", len(files))
print("Expected active sensors:", len(ACTIVE_SENSORS))
print("Expected base features:", len(BASE_FEATURE_COLS))

print("✅ CELL 8 DONE")

Processed files: 16
Expected active sensors: 15
Expected base features: 18
✅ CELL 8 DONE


In [9]:
summary = f"""
NB01 COMPLETED

Datasets: {DATASETS}

Removed Sensors:
{SENSORS_REMOVED}

Active Sensors:
{ACTIVE_SENSORS}

Active Sensor Count: {len(ACTIVE_SENSORS)}

Base Features: {len(BASE_FEATURE_COLS)}

Expected NB02 Engineered Features: 342
"""

with open(LOG_DIR / "nb01_summary.txt", "w") as f:
    f.write(summary)

print(summary)

print("✅ CELL 9 DONE")


NB01 COMPLETED

Datasets: ['FD001', 'FD002', 'FD003', 'FD004']

Removed Sensors:
['sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']

Active Sensors:
['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']

Active Sensor Count: 15

Base Features: 18

Expected NB02 Engineered Features: 342

✅ CELL 9 DONE


In [10]:
zip_path = shutil.make_archive(
    str(OUTPUT_ROOT),
    "zip",
    OUTPUT_ROOT
)

print(f"ZIP CREATED: {zip_path}")

print("🎉 NB01 COMPLETED SUCCESSFULLY")
print("✅ CELL 10 DONE")

ZIP CREATED: /kaggle/working/NB-1-AGAIN-outputs.zip
🎉 NB01 COMPLETED SUCCESSFULLY
✅ CELL 10 DONE
